# Public SEC Covenant Monitoring RAG

A project-specific clone of `02-rag.ipynb`. It retrieves public SEC document records for analyst review of covenant terms, reported leverage, compliance events, amendments, and waivers. The pipeline keeps the course structure: `search`, `build_prompt`, `llm`, and `rag`.

In [ ]:
import os
from functools import partial
from pathlib import Path

from credit_monitoring.ingestion.chunking.evidence import chunk_evidence_documents
from credit_monitoring.ingestion.loaders.sec import build_company_catalog, create_documents
from credit_monitoring.observability.logging import configure_logging
from credit_monitoring.retrieval.lexical.sec import build_index

# Timestamped document/section progress and timings in cell output.
configure_logging()

# Explicit filesystem configuration; reusable module imports have no side effects.
data_dir = Path("data")
if not data_dir.exists():
    data_dir = Path("../data")
html_dir = data_dir / "sec_documents" / "html"
markdown_dir = data_dir / "sec_documents" / "markdown"
source_file = data_dir / "public_sec_documents.csv"


In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()


## 1. Download, clean, convert, and load the selected documents

The loader downloads each HTML filing listed in the selected CSV, saves the original HTML, removes clear page chrome with Beautiful Soup, and converts the cleaned HTML to Markdown. It then selects covenant-related evidence lines with neighboring context and indexes the resulting Markdown chunks with the catalog metadata.

SEC requests need an identifying `SEC_USER_AGENT` environment variable containing the requester's name or organization and a real contact email. Cached HTML and Markdown files can be reused. The first run after adding HTML cleanup reconverts cached HTML so older Markdown does not bypass cleanup.


In [ ]:
# Downloads use your configured identity; cached filings need no network request.
sec_user_agent = os.environ.get("SEC_USER_AGENT", "")


In [ ]:
from markitdown import MarkItDown

converter = MarkItDown(enable_plugins=False)


## 2. Build the Index

Index covenant-related evidence excerpts with issuer and catalog metadata. Each result retains the filing citation and Markdown path for verification.


In [ ]:
filing_documents = create_documents(
    source_file,
    html_dir=html_dir,
    markdown_dir=markdown_dir,
    user_agent=sec_user_agent,
    converter=converter,
)
documents = chunk_evidence_documents(filing_documents, size=3500, step=500)
index = build_index(documents)
print(f"Loaded {len(filing_documents)} filings and {len(documents)} evidence chunks.")
print(f"Original HTML: {html_dir} | Markdown: {markdown_dir}")


In [ ]:
documents[0] if documents else "No evidence excerpts were selected."


## 3. Import and bind the RAG functions

Reusable functions live in `src/credit_monitoring`. Bind the explicit client and index here; prompts are imported from dedicated modules. Narrative responses report source-stated facts without calculating ratios or headroom.


In [ ]:
from credit_monitoring.retrieval.lexical.sec import search as search_records

search = partial(search_records, index=index, document_count=len(documents))


In [ ]:
from credit_monitoring.agents.prompts.assembly import build_prompt
from credit_monitoring.agents.prompts.narrative import NARRATIVE_INSTRUCTIONS as instructions


In [ ]:
from credit_monitoring.application.rag_service import llm as generate_narrative

llm = partial(generate_narrative, client=openai_client)


In [ ]:
from credit_monitoring.application.rag_service import rag as run_rag

rag = partial(run_rag, client=openai_client, index=index, document_count=len(documents))


In [ ]:
search_results = search( """
            For FMC Corporation (FMC), summarize the reported net leverage trend and covenant status in the available SEC records. 
            Cite each record and explain what evidence is still needed to assess current covenant headroom.
        """)

print(search_results)

## 4. Try It Out

Both prompts search the Markdown evidence chunks and include up to two relevant chunks per filing in the LLM context.


### Prompt 1: FMC borrower review


In [ ]:
print(
    rag(
        """
            For FMC Corporation (FMC), summarize the reported net leverage trend and covenant status in the available SEC records. 
            Cite each record and explain what evidence is still needed to assess current covenant headroom.
        """
    )
)


### Prompt 2: Full-corpus comparison table


In [ ]:
print(
    rag(
        """
            Across all indexed SEC filings, compare each issuer's latest disclosed leverage covenant result or status with 
            available prior periods. Present the findings in a concise table for quick analyst review, with columns for issuer, 
            latest period, leverage result, covenant threshold, compliance or waiver status, comparison with prior period, 
            and source citation. Include applicable thresholds and relevant amendments or waivers where the filings support them. 
            Mark unavailable information as "Not disclosed in retrieved sources," and do not infer missing values or status.
        """
    )
)

## 5. Process complete documents and reuse stored facts

Document processing is independent of question-based retrieval. The synchronous service
reads every Markdown section, extracts all relevant facts, and stores successful results
in SQLite. Repeated calls reuse the document/version cache. Automatic verification is
temporarily disabled: result wrappers use schema version 2.1 and `validation: null`.


In [ ]:
from credit_monitoring.application.document_processing_service import DocumentProcessingService

processing_service = DocumentProcessingService(
    client=openai_client,
    source_file=source_file,
    html_dir=html_dir,
    markdown_dir=markdown_dir,
    database_path=data_dir / "processed" / "covenants.sqlite3",
    user_agent=sec_user_agent,
    converter=converter,
)


In [ ]:
# Discover registered documents, including missing or stale extractions; no model call.
[
    (document.document_id, document.status)
    for document in processing_service.list_document_extractions(ticker="FMC")
]


In [ ]:
# One complete filing, including all leverage and interest-coverage schedule rows.
# A cache hit makes no model calls. Explicit force=True requests fresh processing.
fmc_document = processing_service.process_document("DOC_FMC_AMD3_2025")
fmc_extraction = fmc_document.result
print("Cache reused:", fmc_document.cache_hit)
print("Sections completed:", fmc_document.completed_sections, "/", fmc_document.total_sections)
print(fmc_extraction.model_dump_json(indent=2))
print("Verification:", "not run" if fmc_extraction.validation is None else fmc_extraction.validation)


In [ ]:
documents = processing_service.process_all_documents(force=True)

In [ ]:
print(documents[-1].model_dump_json(indent=2))

### Process selected catalog documents

The application can call the same service as soon as a document arrives or requires
reprocessing. No queue or worker is required. This helper processes registered documents
sequentially; invoke it explicitly to avoid starting full-corpus model calls accidentally.


In [ ]:
def process_catalog(ticker=None, force=False):
    return processing_service.process_all_documents(ticker=ticker, force=force)

# Optional: process all registered FMC filings, reusing completed results.
# fmc_documents = process_catalog(ticker="FMC")
# Optional: process the complete registered catalog.
# all_documents = process_catalog()


## 6. Agents SDK with document tools

The OpenAI Agents SDK runs the tool loop. The agent chooses among listing documents,
reading stored extractions, searching source
passages, and processing a document when needed. Each tool result returns to the model;
the model chooses another call or a cited answer. At most six tool rounds execute.
The public trace shows actual calls and outcomes, never private model reasoning.

This question requests complete stored schedules and supporting filing passages. Actual
tool selection remains model-driven; the tests exercise both paths with mocked responses.
The earlier direct `rag()` function remains available independently.


In [ ]:
from credit_monitoring.agents.monitoring import MonitoringAgent

monitoring_agent = MonitoringAgent(client=openai_client, service=processing_service)


In [ ]:
agent_answer = await monitoring_agent.ask_async(
    "Read the complete leverage and interest-coverage schedules in FMC Amendment No. 3. "
    "Include every explicitly named fiscal quarter, then search the FMC filings for "
    "supporting passages and explicitly reported ratios. Cite the sources. "
    "Do not determine which amendment governs or calculate headroom.",
    ticker="FMC",
)
print(agent_answer.answer)
print("Completed:", agent_answer.complete, "Error:", agent_answer.error)


In [ ]:
# Inspect the actual tool execution sequence, including cache and failure outcomes.
for execution in agent_answer.tool_trace:
    print(execution.name, execution.arguments, "ok:", execution.outcome["ok"])
print([source.model_dump() for source in agent_answer.sources])
# The complete structured outcome is also serializable:
# print(agent_answer.model_dump_json(indent=2))
